# MyHeartCounts Data API Demo

This notebook demonstrates how to use the `myheartcounts_ds` package to access Firebase data.

In [ ]:
from datetime import datetime

from myheartcounts_ds import MHC4Client, MHCConfig, User, DiscoveredObservationType

## Configuration

The client can be configured for different environments:

In [ ]:
# Default: production environment
client = MHC4Client()
print(f"Project: {client.config.project_id}")
print(f"Bucket: {client.config.storage_bucket}")

In [ ]:
# Alternative: dev environment
# config = MHCConfig.dev()
# client = MHCClient(config)

# Or custom project
# config = MHCConfig(project_id="my-custom-project")
# client = MHCClient(config)

## List Users

In [ ]:
# Fetch first 5 users
users = client.list_users(limit=5)
print(f"Found {len(users)} users")

In [ ]:
# Display user details
for user in users:
    print(f"User: {user.id}")
    print(f"  Enrolled: {user.date_of_enrollment}")
    print(f"  Group: {user.participant_group}")
    print(f"  Region: {user.us_region}")
    print()

## Get Specific User

In [ ]:
# Get a specific user by ID
user_id = users[0].id if users else "WhTFb0Ok3jeqate1lVSSS8avnPv1"
user = client.get_user(user_id)

if user:
    print(f"User ID: {user.id}")
    print(f"Date of Birth: {user.date_of_birth}")
    print(f"Enrolled: {user.date_of_enrollment}")
    print(f"Last Active: {user.last_active_date}")
    print(f"Participant Group: {user.participant_group}")
    print(f"Height: {user.height_in_cm} cm")
    print(f"Weight: {user.weight_in_kg} kg")
else:
    print("User not found")

## List Observation Types

Discover what observation data types are available in the database. Types are categorized by prefix:
- **HealthKit**: `HK*` identifiers
- **MHC Custom**: `MHC*` identifiers  
- **SensorKit**: `com.apple.SensorKit.*` identifiers

In [ ]:
# Get all observation types for a specific user
user_id = users[1].id if users else "C92dp0MOMpRuNOnPWWbjq5A9M5n2"

all_types = client.list_observation_types(user=user_id)
hk_types = client.list_healthkit_observation_types(user=user_id)
mhc_types = client.list_mhc_observation_types(user=user_id)
sensor_types = client.list_sensorkit_observation_types(user=user_id)

print(f"Observation types for user {user_id}:")
print(f"  Total: {len(all_types)}")
print(f"  HealthKit: {len(hk_types)}")
print(f"  MHC Custom: {len(mhc_types)}")
print(f"  SensorKit: {len(sensor_types)}")

In [ ]:
# Sample observation types across multiple users
all_types = client.list_observation_types(user_limit=10)
hk_types = client.list_healthkit_observation_types(user_limit=10)
mhc_types = client.list_mhc_observation_types(user_limit=10)
sensor_types = client.list_sensorkit_observation_types(user_limit=10)

print("Observation types across sampled users:")
print(f"  Total: {len(all_types)}")
print(f"  HealthKit: {len(hk_types)}")
print(f"  MHC Custom: {len(mhc_types)}")
print(f"  SensorKit: {len(sensor_types)}")

print(f"\nHealthKit types ({len(hk_types)}):")
for t in sorted(hk_types):
    print(f"  - {t}")

print(f"\nMHC Custom types ({len(mhc_types)}):")
for t in sorted(mhc_types):
    print(f"  - {t}")

print(f"\nSensorKit types ({len(sensor_types)}):")
for t in sorted(sensor_types):
    print(f"  - {t}")

## HealthKit Subcategories

HealthKit types can be further filtered by subcategory based on their type identifier prefix:
- **Quantity**: `HKQuantityTypeIdentifier*` (e.g., heart rate, step count)
- **Category**: `HKCategoryTypeIdentifier*` (e.g., sleep analysis, stand hour)
- **Correlation**: `HKCorrelationTypeIdentifier*` (e.g., blood pressure)
- **Workout**: `HKWorkoutTypeIdentifier*`
- **Clinical**: `HKClinicalTypeIdentifier*` (e.g., lab results, medications)
- **Data**: `HKDataType*` (e.g., heartbeat series, state of mind)

In [ ]:
# Get HealthKit subcategory types for a specific user (for reliable verification)
subcategory_user_id = users[0].id if users else "0YEI8vdLURYUidoztxNfUcwhyMb2"

hk_types_user = client.list_healthkit_observation_types(user=subcategory_user_id)
quantity_types = client.list_hk_quantity_observation_types(user=subcategory_user_id)
category_types = client.list_hk_category_observation_types(user=subcategory_user_id)
correlation_types = client.list_hk_correlation_observation_types(user=subcategory_user_id)
workout_types = client.list_hk_workout_observation_types(user=subcategory_user_id)
clinical_types = client.list_hk_clinical_observation_types(user=subcategory_user_id)
data_types = client.list_hk_data_observation_types(user=subcategory_user_id)

print(f"HealthKit subcategory breakdown for user {subcategory_user_id}:")
print(f"  Total HealthKit: {len(hk_types_user)}")
print(f"  Quantity types: {len(quantity_types)}")
print(f"  Category types: {len(category_types)}")
print(f"  Correlation types: {len(correlation_types)}")
print(f"  Workout types: {len(workout_types)}")
print(f"  Clinical types: {len(clinical_types)}")
print(f"  Data types: {len(data_types)}")

print(f"\nQuantity types ({len(quantity_types)}):")
for t in sorted(quantity_types):
    print(f"  - {t}")

print(f"\nCategory types ({len(category_types)}):")
for t in sorted(category_types):
    print(f"  - {t}")

print(f"\nCorrelation types ({len(correlation_types)}):")
for t in sorted(correlation_types):
    print(f"  - {t}")

print(f"\nWorkout types ({len(workout_types)}):")
for t in sorted(workout_types):
    print(f"  - {t}")

print(f"\nClinical types ({len(clinical_types)}):")
for t in sorted(clinical_types):
    print(f"  - {t}")

print(f"\nData types ({len(data_types)}):")
for t in sorted(data_types):
    print(f"  - {t}")

## Retrieve HealthKit Quantity Data

Use `get_hk_quantity()` to retrieve HealthKit quantity observations as a pandas DataFrame. The method:
- Accepts a `DiscoveredObservationType` enum value (must be `HK_QUANTITY_*`)
- Supports optional time filtering with `start_time` and `end_time`
- Returns a DataFrame with timestamps, values, units, and device metadata

In [ ]:
# Get heart rate data for a user
hr_user_id = users[1].id if users else "0YEI8vdLURYUidoztxNfUcwhyMb2"

# Retrieve all heart rate observations
df = client.get_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id=hr_user_id,
)

print(f"Heart rate records for user {hr_user_id}: {len(df)}")
print(f"\nDataFrame columns: {list(df.columns)}")
print(f"\nFirst few records:")
df.head()

In [ ]:
user_id = "0YEI8vdLURYUidoztxNfUcwhyMb2"
from myheartcounts_ds.constants import HEALTH_OBSERVATION_COLLECTION_PREFIX                                                                                                                                                    
import json

user_id = hr_user_id
obs_type = DiscoveredObservationType.HK_QUANTITY_STEP_COUNT

collection_name = f"{HEALTH_OBSERVATION_COLLECTION_PREFIX}{obs_type.value}"
print(f"Collection: {collection_name}")

collection_ref = client.db.collection("users").document(user_id).collection(collection_name)

# Try without limit - just get first doc manually
count = 0
for doc in collection_ref.stream():
    data = doc.to_dict()
    print(json.dumps(data, indent=2, default=str))
    count += 1
    if count >= 1:
        break

print(f"Total docs seen: {count}")

# If this still shows nothing, let's verify the exact values being used:

print(f"user_id: '{user_id}'")
print(f"obs_type.value: '{obs_type.value}'")
print(f"collection_name: '{collection_name}'")

# Double-check get_hk_quantity actually returns data
df_test = client.get_hk_quantity(obs_type, user_id=user_id)
print(f"get_hk_quantity returned {len(df_test)} rows")

In [ ]:
# Filter by time range
df_filtered = client.get_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_STEP_COUNT,
    user_id=hr_user_id,
    start_time=datetime(2024, 1, 1),
    end_time=datetime(2025, 1, 1),
)

print(f"Step count records in 2024: {len(df_filtered)}")

if not df_filtered.empty:
    print(f"\nDate range: {df_filtered['start_time'].min()} to {df_filtered['start_time'].max()}")
    print(f"Total steps: {df_filtered['value'].sum():,.0f}")
    print(f"Average per record: {df_filtered['value'].mean():,.1f}")
    print(f"Unit: {df_filtered['unit'].iloc[0]}")
    
    # Show unique data sources
    print(f"\nData sources:")
    for source in df_filtered['source_name'].dropna().unique():
        print(f"  - {source}")

In [ ]:
# List all available HK_QUANTITY_* types from the enum
print("Available HK_QUANTITY_* types in DiscoveredObservationType:")
for member in DiscoveredObservationType:
    if member.name.startswith("HK_QUANTITY_"):
        print(f"  DiscoveredObservationType.{member.name}")
        print(f"    -> {member.value}")